# Generation, Sampling, And Evaluation

Once a model has learned logits, inference is the problem of turning those logits into a sequence one token at a time. This notebook makes the generation loop, sampling controls, and perplexity interpretation explicit.


## Problem: What Must This Component Compute?

During generation, the model repeatedly maps an integer token tensor of shape `(B, T)` to logits of shape `(B, T, V)` and samples the next token from the final time step without leaking future information.

We will first write the operation directly with tensors so every dimension is visible. Only after the numerical checks pass will we wrap the operation in a reusable module.

Why this matters later: when an architecture paper claims to improve decoding, evaluation, or controllability, it is usually changing one of these constraints: how logits are filtered, how uncertainty is shaped, how context is truncated, or how outputs are judged.


## 1. Autoregressive generation loop.

Autoregressive decoding repeats one simple loop: run the model on the current context, read the final-step logits, sample one token, append it, and continue. If `idx` has shape `(B, T)`, then after one step the next-token sample has shape `(B, 1)` and concatenation produces `(B, T + 1)`.


In [ ]:
import math
import torch
import torch.nn.functional as F

from llm_from_scratch.configs import ModelConfig, set_seed
from llm_from_scratch.evaluate import perplexity
from llm_from_scratch.generate import generate, sample_next_token, top_k_filter, top_p_filter
from llm_from_scratch.model import TinyGPT

set_seed(123)
logits = torch.tensor([[2.0, 1.0, 0.5, -1.0]])
scaled = logits / 0.7
top_k_logits = top_k_filter(scaled, 2)
top_p_logits = top_p_filter(scaled, 0.8)
next_token = sample_next_token(logits, temperature=0.7, top_k=2, top_p=0.9)

assert top_k_logits.shape == logits.shape
assert torch.isinf(top_k_logits[0, 2:]).all()
assert next_token.shape == (1, 1)

{
    "top_k_logits": top_k_logits.tolist(),
    "top_p_logits": top_p_logits.tolist(),
    "sampled_token": int(next_token.item()),
}


## 2. Temperature.

Temperature rescales logits before softmax. Values below `1.0` sharpen the distribution, making high-probability tokens more dominant. Values above `1.0` flatten the distribution and increase randomness.


## 3. Top-k filtering.

Top-k keeps only the `k` largest logits and masks the rest to negative infinity before softmax. This enforces a fixed candidate set size regardless of how concentrated or diffuse the model distribution is.


## 4. Top-p filtering.

Top-p keeps the smallest prefix of tokens whose cumulative probability exceeds `p`. Unlike top-k, the number of retained candidates changes with the shape of the model distribution.


In [ ]:
import torch
from llm_from_scratch.configs import ModelConfig
from llm_from_scratch.generate import generate
from llm_from_scratch.model import TinyGPT

cfg = ModelConfig(vocab_size=20, block_size=8, n_embd=16, n_head=4, n_layer=1)
model = TinyGPT(cfg)
idx = torch.zeros((1, 3), dtype=torch.long)
generate(model, idx, max_new_tokens=5, temperature=1.0, top_k=5)


## 5. Perplexity.

Perplexity is the exponential of average negative log-likelihood:

\[
\text{perplexity} = \exp(\text{loss}).
\]

Lower perplexity means the model assigns higher probability to the observed next tokens on average.


## 6. Qualitative generation checks.

Qualitative checks complement perplexity by asking whether generations stay on topic, respect prompt prefixes, and avoid obvious degeneration like loops or impossible future leakage. They are not a substitute for metrics, but they do catch failure modes a scalar can hide.


In [ ]:
loss_value = 1.5
ppl = perplexity(loss_value)

set_seed(123)
cfg = ModelConfig(vocab_size=20, block_size=8, n_embd=16, n_head=4, n_layer=1, dropout=0.0)
model = TinyGPT(cfg)
prompt = torch.tensor([[0, 1, 2]], dtype=torch.long)
cold = generate(model, prompt.clone(), max_new_tokens=4, temperature=0.7, top_k=3)
set_seed(123)
warm = generate(model, prompt.clone(), max_new_tokens=4, temperature=1.3, top_p=0.9)

assert abs(ppl - math.exp(loss_value)) < 1e-9
assert cold.shape == warm.shape == (1, 7)

{
    "perplexity": ppl,
    "cold_tokens": cold.tolist(),
    "warm_tokens": warm.tolist(),
}


In [ ]:
set_seed(123)
cfg = ModelConfig(vocab_size=20, block_size=8, n_embd=16, n_head=4, n_layer=1, dropout=0.0)
model = TinyGPT(cfg)
idx = torch.tensor([[0, 1, 2, 3]], dtype=torch.long)
targets = torch.tensor([[1, 2, 3, 4]], dtype=torch.long)
logits, loss = model(idx, targets)

assert logits.shape == (1, 4, cfg.vocab_size)
assert loss is not None and torch.isfinite(loss)

{
    "logits_shape": tuple(logits.shape),
    "loss": float(loss.item()),
    "last_step_probs": F.softmax(logits[0, -1], dim=-1)[:5].tolist(),
}


## Abstraction Bridge

The package helper `generate` packages the exact loop described above: truncate to the model block size, run `TinyGPT`, apply temperature and optional filters, sample, and append. `perplexity` then translates average loss into a more interpretable branching-factor style metric. The high-level API is small because the underlying tensor operations are already doing the real work.


## Exercise Checkpoint 1

1. Why does the generation loop sample from `logits[:, -1, :]` instead of from all time steps at once?
2. What shape does `torch.cat((idx, next_token), dim=1)` produce if `idx` is `(B, T)` and `next_token` is `(B, 1)`?


## Exercise Checkpoint 2

1. When would top-p sampling keep more candidates than top-k, and why?
2. Why can perplexity improve even when short generations still look qualitatively weak?
